# BıRAG — taban ↔ ince ayar: üretim defteri

Aynı soruları iki modele sorar, cevapları Drive'a yazar. **Puanlama burada yapılmaz**, veri makinesinde yapılır.

| kol | model |
|---|---|
| **T0** | `unsloth/gemma-4-26B-A4B-it`, eğitilmemiş |
| **T1** | aynı model + `v0.1.1` ile eğittiğiniz LoRA adapter |

İki kol **aynı yüklenmiş ağırlıkları** kullanır. Önce T0 üretilir, sonra adapter takılıp T1 üretilir; iki kol arasında değişen tek şey adapter.

*Çalışma zamanı → Çalışma zamanı türünü değiştir → **A100 (80 GB)***. 2. hücrede `ADAPTER_DIR`'i kontrol edin, sonra *Tümünü çalıştır*.
Oturum koparsa yeniden *Tümünü çalıştır*. Biten işler atlanır, yarım kalan set kaldığı yerden sürer.

⛔ **Ön kayıt:** `configs/deney/2026-10-12-taban-vs-ince-on-kayit.json`. Setler, üretim ayarı ve karar kuralı orada mühürlü.
Bu defterde 2. hücre dışında hiçbir şeyi değiştirmeyin; üretim ayarı ön kayıttan okunur.
Plan: `docs/taban-vs-ince-ayar-degerlendirme-plani.md`.

⛔ **Kriz:** eğitim verisinde kriz dilimi yok. Güvenlik seti, ince ayarın tabanın kriz davranışını bozup bozmadığını ölçer;
modelin krize hazır olduğunu göstermez.

### 1. Drive ve kurulum

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!pip install -q unsloth

### 2. Parametreler
Yalnız bu hücre değişir.

In [ ]:
MODEL_ID    = "unsloth/gemma-4-26B-A4B-it"
MODEL_KISA  = "26b"
ADAPTER_DIR = "/content/drive/MyDrive/birag-sohbet/lora-26b"   # v0.1.1 adapter'ı: adapter_config.json'ın olduğu klasör
TEST_IDLER  = "/content/drive/MyDrive/birag-sohbet/test_idler.json"   # eğitimde ayırdığınız test kayıtları (yoksa yeniden hesaplanır)
DRIVE       = "/content/drive/MyDrive/birag-taban-vs-ince"
REPO        = "/content/birag-data-generation"
ON_KAYIT    = "configs/deney/2026-10-12-taban-vs-ince-on-kayit.json"
BATCH       = 8      # aynı anda üretilen soru sayısı. Hata ya da bellek taşması olursa 4 ya da 1 yapın

### 3. Depo ve ön kayıt
Eval setleri, veri ve yardımcı kod depodan gelir. Mühürlü dosyalardan biri ön kayıttakinden farklıysa defter durur.

In [ ]:
import os, sys, json, hashlib, subprocess, random, time
from pathlib import Path
from datetime import datetime

if not os.path.exists(REPO):
    !git clone -q https://github.com/enveryildirim/birag-data-generation.git {REPO}
sys.path.insert(0, f"{REPO}/src")
GIT_REV = subprocess.check_output(["git", "-C", REPO, "rev-parse", "HEAD"], text=True).strip()

OK = json.loads(Path(REPO, ON_KAYIT).read_text(encoding="utf-8"))
sha16 = lambda p: hashlib.sha256(Path(REPO, p).read_bytes()).hexdigest()[:16]
bozuk = {p: (s, sha16(p)) for p, s in OK["muhurlu_dosyalar"].items() if sha16(p) != s}
assert not bozuk, f"⛔ mühürlü dosya ön kayıttakinden farklı — durun, bildirin: {bozuk}"
assert MODEL_ID == OK["model_id"], f"⛔ ön kayıttaki model {OK['model_id']}"
U = OK["uretim"]
oku = lambda yol: [json.loads(x) for x in open(Path(REPO, yol), encoding="utf-8") if x.strip()]
print(f"depo {GIT_REV[:7]} · mühür sağlam ({len(OK['muhurlu_dosyalar'])} dosya)")
print(f"üretim: düşünme açık · en fazla {U['max_new_tokens']} jeton · birincil açgözlü · ikincil {U['ornekleme']} × {U['k']}")

### 4. Taban modeli yükle
26B-A4B bf16 ~52 GB tutar; 4-bit **kullanılmaz** (çıktılar eğitilen modelden sapar).

In [ ]:
os.environ["UNSLOTH_RETURN_LOGITS"] = "1"      # test kaybı için logit gerekir
from unsloth import FastModel
import torch

bellek = torch.cuda.get_device_properties(0).total_memory / 1e9
assert MODEL_KISA != "26b" or bellek > 70, f"⛔ GPU {bellek:.0f} GB — 26B-A4B için A100 80GB seçin"
model, tokenizer = FastModel.from_pretrained(MODEL_ID, max_seq_length=4096, dtype=torch.bfloat16, load_in_4bit=False)
FastModel.for_inference(model)
tok = getattr(tokenizer, "tokenizer", tokenizer)
tok.padding_side = "left"

try:
    from huggingface_hub import model_info
    REV = model_info(MODEL_ID).sha
except Exception as e:
    REV = None; print("⚠️ model revizyonu okunamadı:", e)
EOS = model.generation_config.eos_token_id
EOS = sorted(set((list(EOS) if isinstance(EOS, (list, tuple)) else [EOS]) + [tok.convert_tokens_to_ids("<turn|>")]))
PAD = tok.pad_token_id if tok.pad_token_id is not None else EOS[0]
print(f"GPU {torch.cuda.get_device_name(0)} ({bellek:.0f} GB) · {MODEL_ID} @ {str(REV)[:12]} · EOS {EOS}")

### 5. İstem ve üretim
- Düşünme `enable_thinking=True` ile açılır; sistem istemine elle `<|think|>` **eklenmez**. Her istem denetlenir: `<|think|>` tam bir kez geçmeli, istem `<|turn>model\n` ile bitmeli. Bitmiyorsa şablon boş bir düşünce bloğu eklemiş, yani düşünme kapanmış demektir.
- Geçmişteki asistan turlarına yalnız görünen cevap konur.
- Biten her parti hemen Drive'a yazılır (`yarim/`). Partiler sabit sırayla kurulduğu için oturum koparsa aynı partiler yeniden üretilir.

In [ ]:
from golden_eval import cikti_ayir
import importlib.metadata as md

def istem(messages):
    msgs = [{"role": m["role"], "content": m["content"]} for m in messages]
    s = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=True)
    assert s.count("<|think|>") == 1, "⛔ düşünme işareti tam bir kez olmalı"
    assert s.endswith("<|turn>model\n"), f"⛔ düşünme açılmadı, istemin sonu: {s[-60:]!r}"
    return s

def uret_toplu(istemler, ornekle=False, tohum=0, k=1):
    x = tok(istemler, add_special_tokens=False, return_tensors="pt", padding=True).to(model.device)
    if ornekle:
        torch.manual_seed(tohum)
        ayar = dict(do_sample=True, num_return_sequences=k, **U["ornekleme"])
    else:
        ayar = dict(do_sample=False)
    t0 = time.time()
    with torch.no_grad():
        y = model.generate(**x, max_new_tokens=U["max_new_tokens"], eos_token_id=EOS, pad_token_id=PAD, **ayar)
    sure = (time.time() - t0) / len(y)
    cikti = []
    for satir in y[:, x["input_ids"].shape[1]:].tolist():
        n = next((i for i, t in enumerate(satir) if t in EOS), None)
        yeni = satir if n is None else satir[:n]
        ham = tok.decode(yeni, skip_special_tokens=False)
        dus, cevap, kapandi = cikti_ayir(ham)
        cikti.append({"ham": ham, "thinking": dus, "cevap": cevap, "thinking_kapandi": kapandi,
                      "sure_sn": round(sure, 1), "uretim_token": len(yeni) + int(n is not None),
                      "kesildi": n is None})
    return cikti

YARIM = Path(DRIVE, "yarim"); YARIM.mkdir(parents=True, exist_ok=True)

def kos(etiket, isler, ornekle=False):
    # isler: [(anahtar, istem, ek_alanlar)] — biten partiler atlanır, yarım parti baştan üretilir.
    # ornekle=True: her istem için ön kayıttaki tohumla k örnek (ek_alanlar["tohum"])
    yol = YARIM / f"{etiket}.jsonl"
    bitti = {}
    if yol.exists():
        for x in open(yol, encoding="utf-8"):
            r = json.loads(x); bitti[r["_anahtar"]] = r
    adim = 1 if ornekle else BATCH
    for b in range(0, len(isler), adim):
        parca = isler[b:b + adim]
        if all(p[0] in bitti for p in parca): continue
        if ornekle:
            (anahtar, s, ek), = parca
            cikti = [{"ornekler": uret_toplu([s], ornekle=True, tohum=ek["tohum"], k=U["k"])}]
        else:
            cikti = uret_toplu([p[1] for p in parca])
        with open(yol, "a", encoding="utf-8") as f:
            for (anahtar, _, ek), c in zip(parca, cikti):
                r = {"_anahtar": anahtar, **ek, **c}
                bitti[anahtar] = r
                f.write(json.dumps(r, ensure_ascii=False) + "\n")
        print(f"\r  {etiket}: {sum(p[0] in bitti for p in isler)}/{len(isler)}", end="")
    print(f"\r  {etiket}: {len(isler)}/{len(isler)} ✅")
    return [bitti[p[0]] for p in isler]

def surum(p):
    try: return md.version(p)
    except md.PackageNotFoundError: return None

def ortam_kur():
    o = {"kosucu": "notebooks/taban-vs-ince-ayar-uretim.ipynb", "on_kayit": ON_KAYIT, "git_rev": GIT_REV,
         "kol": KOL, "model_id": MODEL_ID, "model_revizyon": REV, "batch": BATCH, "uretim": U,
         "surumler": {p: surum(p) for p in ("torch", "transformers", "peft", "unsloth", "unsloth_zoo", "accelerate")},
         "gpu": torch.cuda.get_device_name(0), "cuda": torch.version.cuda, "python": sys.version.split()[0]}
    if KOL == "T1":
        o["adapter"] = ADAPTER_DIR
        o["adapter_sha256_16"] = hashlib.sha256(Path(ADAPTER_DIR, "adapter_model.safetensors").read_bytes()).hexdigest()[:16]
        o["adapter_config"] = json.loads(Path(ADAPTER_DIR, "adapter_config.json").read_text())
    return o

def bitmis(alt, kisa):
    return any(Path(DRIVE, alt).glob(f"*-{MODEL_KISA}-{KOL}-{kisa}/kosu.json"))

def yaz(alt, kisa, set_yol, satirlar, **ek):
    etiket = f"{MODEL_KISA}-{KOL}-{kisa}"
    d = Path(DRIVE, alt, f"{datetime.now():%Y%m%d-%H%M%S}-{etiket}"); d.mkdir(parents=True)
    satir = "".join(json.dumps({k: v for k, v in r.items() if not k.startswith("_")}, ensure_ascii=False) + "\n"
                    for r in satirlar)
    (d / "sonuclar.jsonl").write_text(satir, encoding="utf-8")
    (d / "ham.jsonl").write_text(satir, encoding="utf-8")
    (d / "kosu.json").write_text(json.dumps({
        "etiket": etiket, "kol": KOL, "set": set_yol, "set_sha256_16": sha16(set_yol) if set_yol else None,
        "model": MODEL_ID, "adapter": ADAPTER_DIR if KOL == "T1" else None, "thinking": True,
        "max_tokens": U["max_new_tokens"], "uretim_sn": round(sum(r.get("sure_sn", 0) + sum(x["sure_sn"] for x in r.get("ornekler", [])) for r in satirlar)),
        "judge_sn": 0, "tarih": datetime.now().isoformat(timespec="seconds"), **ek}, ensure_ascii=False, indent=1))
    (d / "ortam.json").write_text(json.dumps(ORTAM, ensure_ascii=False, indent=1, default=str))
    print("  📁", d.name)

### 6. Duman testi
Bütün istemler kurulur ve denetlenir; model çalışmaz. Sonra 3 soru üretilir ve toplam süre tahmin edilir.
`düşünme` satırı boşsa ya da cevap kesilmişse **durun, bildirin**.

In [ ]:
SETLER = {s["kisa"]: s for s in OK["setler"]}
n_istem = 0
for s in OK["setler"]:
    for o in oku(s["yol"]):
        istem(o["messages"]); n_istem += 1
print(f"✅ {n_istem} istem kuruldu ve denetlendi")

ornek = [oku(SETLER[k]["yol"])[0] for k in ("safety", "forget", "golden_dev")]
c = uret_toplu([istem(o["messages"]) for o in ornek])
for o, r in zip(ornek, c):
    print(f"\n── {o['id']} · {r['uretim_token']} jeton · kesildi={r['kesildi']}")
    print("düşünme:", r["thinking"][:200].replace("\n", " "))
    print("cevap  :", r["cevap"][:300].replace("\n", " "))
toplam = OK["beklenen_uretim_kol_basina"] + len(oku(SETLER[OK["ornekleme_seti"]]["yol"])) * U["k"]
print(f"\n⏱️ üretim başına ~{c[0]['sure_sn']} sn (3'lü parti) · kol başına {toplam} üretim → bir kol ≈ "
      f"{toplam * c[0]['sure_sn'] / 3600:.1f} saat, iki kol ≈ {2 * toplam * c[0]['sure_sn'] / 3600:.1f} saat (BATCH={BATCH} ile muhtemelen daha kısa)")

### 7. Kolu üret
Bir kol için sırayla şunlar üretilir:
1. tek turlu setler (güvenlik · unutma · bağlam ×3 · dalkavukluk · golden dev/test)
2. çok turlu konuşmalar
3. ayrılmış test bölmesi (kayıp ve üretim)
4. güvenlik setinin 5'er örneklemesi

In [ ]:
import torch.nn.functional as F

def resmi_bicim(r):
    out = []
    for m in r["messages"]:
        d = {"role": m["role"], "content": m["content"]}
        if m["role"] == "assistant" and m.get("thinking"):
            d["reasoning_content"] = m["thinking"]
        out.append(d)
    return out

def test_kayitlari():
    u = OK["uyum"]
    kayit = oku(u["veri"])
    karisik = kayit[:]; random.Random(u["tohum"]).shuffle(karisik)
    n = round(len(karisik) * u["ayrilan_oran"])
    test = karisik[n // 2:n]
    if os.path.exists(TEST_IDLER):
        dosya = json.load(open(TEST_IDLER)); dosya = dosya if isinstance(dosya, list) else dosya.get("test", dosya)
        assert sorted(dosya) == sorted(r["id"] for r in test), "⛔ test_idler.json yeniden hesaplanan bölmeyle aynı değil — durun, bildirin"
        print(f"  test bölmesi: {len(test)} kayıt · Drive'daki test_idler.json ile aynı")
    else:
        print(f"  ⚠️ {TEST_IDLER} yok — bölme ön kayıttaki kurala göre yeniden hesaplandı ({len(test)} kayıt)")
    return test

def son_tur_kaybi(r):
    tam = tok.apply_chat_template(resmi_bicim(r), tokenize=False, enable_thinking=bool(r.get("has_thinking")))
    i = tam.rfind("<|turn>model\n") + len("<|turn>model\n")
    on = tok(tam[:i], add_special_tokens=False)["input_ids"]
    ids = tok(tam, add_special_tokens=False)["input_ids"]
    assert ids[:len(on)] == on, "⛔ önek hizası bozuk"
    x = torch.tensor([ids], device=model.device)
    with torch.no_grad():
        logits = model(input_ids=x).logits
    assert logits.shape[1] == len(ids), "⛔ model logit döndürmedi — durun, bildirin"
    hedef = x[0, len(on):]
    return F.cross_entropy(logits[0, len(on) - 1:-1].float(), hedef).item(), len(hedef)

def kol_kos():
    print(f"══ {KOL} ══")
    # 1. tek tur
    for s in OK["setler"]:
        if s.get("uretimi"):                 # aynı istemler, başka cetvel: üretim kopyalanır
            continue
        if bitmis(s["alt"], s["kisa"]):
            print(f"  {s['kisa']}: zaten var"); continue
        ogeler = oku(s["yol"])
        satirlar = kos(f"{MODEL_KISA}-{KOL}-{s['kisa']}", [(o["id"], istem(o["messages"]), {"id": o["id"]}) for o in ogeler])
        if s["alt"] == "golden-kosu":
            for r in satirlar: r["judge"] = None
        yaz(s["alt"], s["kisa"], s["yol"], satirlar, oge=len(satirlar))
    for s in OK["setler"]:
        if not s.get("uretimi") or bitmis(s["alt"], s["kisa"]):
            continue
        kaynak = SETLER[s["uretimi"]]
        d = sorted(Path(DRIVE, kaynak["alt"]).glob(f"*-{MODEL_KISA}-{KOL}-{kaynak['kisa']}"))[-1]
        eski = {json.loads(x)["id"]: json.loads(x) for x in open(d / "ham.jsonl", encoding="utf-8")}
        satirlar = [{**eski[o[s["id_alani"]]], "id": o["id"]} for o in oku(s["yol"])]
        yaz(s["alt"], s["kisa"], s["yol"], satirlar, oge=len(satirlar), uretim_kaynagi=d.name)

    # 2. çok tur — her turda 10 konuşma birlikte; geçmişe yalnız cevap
    ct = OK["cok_turlu"]
    if bitmis("cok-turlu-kosu", "cokturlu"):
        print("  cokturlu: zaten var")
    else:
        konusma = oku(ct["set"]); turlar = {}
        for k in range(1, max(len(o["kullanici_turlari"]) for o in konusma) + 1):
            isler = []
            for o in konusma:
                if k > len(o["kullanici_turlari"]): continue
                msgs = [dict(m) for m in o["messages"]]
                for j in range(1, k + 1):
                    msgs.append({"role": "user", "content": o["kullanici_turlari"][j - 1]})
                    if j < k: msgs.append({"role": "assistant", "content": turlar[(o["id"], j)]["cevap"]})
                isler.append((f"{o['id']}-t{k}", istem(msgs),
                              {"konusma": o["id"], "tur": k, "kullanici": o["kullanici_turlari"][k - 1]}))
            for r in kos(f"{MODEL_KISA}-{KOL}-cokturlu", isler):
                turlar[(r["konusma"], r["tur"])] = r
        satirlar = [turlar[k] for k in sorted(turlar)]
        yaz("cok-turlu-kosu", "cokturlu", ct["set"], satirlar, konusma=len(konusma), tur=len(satirlar))

    # 3. ayrılmış test bölmesi
    if bitmis("uyum-kosu", "uyum"):
        print("  uyum: zaten var")
    else:
        test = test_kayitlari()
        kayip = []
        for i, r in enumerate(test, 1):
            l, n = son_tur_kaybi(r); kayip.append({"id": r["id"], "kayip": l, "n_token": n})
            print(f"\r  kayıp: {i}/{len(test)}", end="")
        ort = sum(x["kayip"] * x["n_token"] for x in kayip) / sum(x["n_token"] for x in kayip)
        print(f"\r  test kaybı (son asistan turu, jeton ağırlıklı): {ort:.4f}")
        satirlar = kos(f"{MODEL_KISA}-{KOL}-uyum",
                       [(r["id"], istem(r["messages"][:-1]),
                         {"id": r["id"], "ref_cevap": r["messages"][-1]["content"], "has_thinking": r.get("has_thinking")})
                        for r in test])
        kmap = {x["id"]: x for x in kayip}
        for s in satirlar: s.update(kayip=kmap[s["id"]]["kayip"], n_token=kmap[s["id"]]["n_token"])
        yaz("uyum-kosu", "uyum", OK["uyum"]["veri"], satirlar, kayit=len(satirlar), test_kaybi=ort)

    # 4. ikincil: güvenlik setinin örneklemesi
    o_set = OK["ornekleme_seti"]
    if bitmis("ornek-kosu", "ornek"):
        print("  ornek: zaten var")
    else:
        ogeler = oku(SETLER[o_set]["yol"])
        satirlar = kos(f"{MODEL_KISA}-{KOL}-ornek",
                       [(o["id"], istem(o["messages"]), {"id": o["id"], "tohum": U["ornek_tohum_tabani"] + i})
                        for i, o in enumerate(ogeler)], ornekle=True)
        yaz("ornek-kosu", "ornek", SETLER[o_set]["yol"], satirlar, oge=len(satirlar), k=U["k"])

### 8. T0 — taban

In [ ]:
KOL = "T0"
assert not hasattr(model, "peft_config"), "⛔ T0'da adapter takılı olmamalı — çalışma zamanını yeniden başlatın"
ORTAM = ortam_kur()
kol_kos()

### 9. Adapter'ı tak
Adapter, 4. hücrede yüklenen **aynı** taban ağırlıklarının üstüne takılır. `adapter_config.json`'daki taban model `MODEL_ID` ile aynı olmalı.

In [ ]:
from peft import PeftModel

cfg = json.loads(Path(ADAPTER_DIR, "adapter_config.json").read_text())
taban = str(cfg.get("base_model_name_or_path", "")).rstrip("/").split("/")[-1].lower()
assert taban == MODEL_ID.split("/")[-1].lower(), f"⛔ adapter başka bir taban için eğitilmiş: {cfg.get('base_model_name_or_path')}"
if not hasattr(model, "peft_config"):
    model = PeftModel.from_pretrained(model, ADAPTER_DIR)
model.eval(); FastModel.for_inference(model)
n_lora = sum(1 for _, m in model.named_modules() if hasattr(m, "lora_A") and len(m.lora_A))
assert n_lora > 0, "⛔ LoRA modülü bulunamadı"
print(f"✅ adapter takıldı · {n_lora} LoRA modülü · r {cfg.get('r')} · alpha {cfg.get('lora_alpha')}")

### 10. T1 — ince ayar

In [ ]:
KOL = "T1"
ORTAM = ortam_kur()
kol_kos()

### 11. Paket
İki kol tamamsa Drive'a tek bir zip yazılır: `birag-taban-vs-ince/taban-vs-ince-26b.zip`. Zip'i veri makinesine getirin.

In [ ]:
import zipfile
ALTLAR = sorted({s["alt"] for s in OK["setler"]} | {"cok-turlu-kosu", "uyum-kosu", "ornek-kosu"})
beklenen = [s["kisa"] for s in OK["setler"]] + ["cokturlu", "uyum", "ornek"]
eksik = [f"{kol}-{k}" for kol in ("T0", "T1") for k in beklenen
         if not any(Path(DRIVE).glob(f"*/*-{MODEL_KISA}-{kol}-{k}/kosu.json"))]
print(f"tamam: {2 * len(beklenen) - len(eksik)}/{2 * len(beklenen)}", "· eksik:", eksik or "yok")
if not eksik:
    paket = Path(DRIVE, f"taban-vs-ince-{MODEL_KISA}.zip")
    with zipfile.ZipFile(paket, "w", zipfile.ZIP_DEFLATED) as z:
        for alt in ALTLAR:
            for f in Path(DRIVE, alt).rglob("*"):
                if f.is_file() and f"-{MODEL_KISA}-" in str(f): z.write(f, f.relative_to(DRIVE))
    print("✅ paket:", paket, f"{paket.stat().st_size / 1e6:.1f} MB")